# Libraries

In [3]:
import pandas as pd
import os

In [4]:
dataset_path = os.path.join('..', 'datasets', 'Philippine Fake News Corpus.csv')
df = pd.read_csv(dataset_path)

# Data Processing

In our exploratory data analysis (EDA), we've identified the columns that we will be working with, namely, Content, and Brand. However, before training the actual model, we'll need to format the data in numbers since that's how the model can understand and learn the patterns.

We'll create a copy of the original dataframe just in case we want a reference to the original dataset, then use the copy for any modifications and processing that we'll perform.

In [5]:
news_df = df[['Brand','Content','Label']].copy()
news_df

,Brand,Content,Label
0,Inquirer,Pollution caused by traditional cooking fuel i...,Credible
1,Manila Times,Justice Secretary Vitaliano Aguirre 2nd and Ph...,Credible
2,Inquirer,President Rodrigo Duterte on Monday night desc...,Credible
3,Manila Times,THE militant fisher folk group Pambansang Laka...,Credible
4,Inquirer,Magdalo Rep. Gary Alejano is willing to lead t...,Credible
...,...,...,...
22453,Get Real Philippines,"Indeed, everybody is shocked — just shocked! —...",Not Credible
22454,Manila Times,"A TOTAL of 132,259 individuals from 28,101 fam...",Credible
22455,Adobo Chronicles,Shortly after Rod Duterte announced there will...,Not Credible
22456,Adobo Chronicles,President Barack Obama met for the first time ...,Not Credible


There are 2 things we need to address in our dataset:
- Class Imbalance - The Credible label is twice as large as the Not Credible label which may introduce label bias to the model where they predict "Credible" for majority of the dataset.

- String to Number - The model can't understand strings, so, we'll have to convert the strings to numbers, where each word or symbol is a unique number.

## Addressing the Class Imbalance

In [6]:
news_df.Label.value_counts()

Label
Credible        14802
Not Credible     7656
Name: count, dtype: int64

Let's first begin by defining the majority class and the minority class.

In [18]:
majority = news_df[news_df['Label'] == 'Credible']
minority = news_df[news_df['Label'] == 'Not Credible']
print(f'The majority has a length of {len(majority)} while the minority has {len(minority)}')
print(f'The difference between the two is {len(majority) - len(minority)}. The minority is about {round(len(minority)/len(majority) * 100, 2)}% of the majority')

The majority has a length of 14802 while the minority has 7656
The difference between the two is 7146. The minority is about 51.72% of the majority


To resolve this, we can upsample the minority class to match that of the majority class, then concatenate them.

In [31]:
news_df_upsampled = pd.concat([
    majority,
    minority.sample(len(majority), replace = True)
]).sample(frac = 1, random_state = 42).reset_index(drop = True) # Shuffle Dataset

news_df_upsampled.Label.value_counts()

Label
Not Credible    14802
Credible        14802
Name: count, dtype: int64

In [32]:
news_df_upsampled.head()

,Brand,Content,Label
0,Duterte Daily Stories,One of the high profile inmates and the most c...,Not Credible
1,Adobo Chronicles,Vote-buying is not a new phenomenon in the Phi...,Not Credible
2,Inquirer,The Philippine Drug Enforcement Agency (PDEA) ...,Credible
3,Manila Times,Low-cost carrier Cebu Pacific said it would di...,Credible
4,Get Real Philippines,Topping the oddities that happened over the pa...,Not Credible


## Addressing String Encoding (Number Conversion)